<a href="https://colab.research.google.com/github/rcsb/rcsb-training-resources/blob/master/training-events/2026/python-rcsb-api/models_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using `rcsb-api` to access RCSB PDB's Model Server API

In [ ]:
# Install `rcsb-api`
%pip install --upgrade rcsb-api

## Model Server API Query Basics
The Model Server API ([models.rcsb.org](http://models.rcsb.org)) can be used to fetch specific sets of coordinate data from PDB structures, including:
- Fetch the coordinates of specific entities, chains, or chemical components
- Fetch the coordinates of particular ligands within a structure
- Fetch just the surrounding residues or ligands within a proximal distance of a particular residue or ligand

For a full list of available methods, see the [*rcsb-api* documentation](https://rcsbapi.readthedocs.io/en/latest/model_api/quickstart.html#query-types) and [Model Server API specification](http://models.rcsb.org).

#### ***Important:*** *This service should not be used for bulk download of coordinate data – use rsync instead!* (see: https://wwpdb.org/ftp/pdb-ftp-sites).


## Model API Query Construction

### Import `ModelQuery`
```python
from rcsbapi.model import ModelQuery
```

### Query Methods
The `ModelQuery` object supports the following types of queries/methods:

| Method                        | Description                                                                                                                                           |
| ----------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------- |
| `.get_ligand()`               | Retrieve ligand coordinates from a given structure                                                                                                    |
| `.get_atoms()`                | Fetch the coordinates of any part of a given structure (e.g., a particular entity, chain, or ligand)                                                  |
| `.get_residue_interaction()`  | Retrieve the coordinates of all residues and ligands within a specified distance from a given residue or ligand (takes crystal symmetry into account) |
| `.get_residue_surroundings()` | Retrieve the coordinates of all residues and ligands within a specified distance from a given residue or ligand (ignores crystal symmetry)            |
| `.get_surrounding_ligands()`  | Retrieve the coordinates of all ligands within a specified distance from a given residue or ligand (taking crystal symmetry into account)             |
| `.get_assembly()`             | Extract the coordinates of a structural assembly (selected group of instances or “chains”) from an entry                                              |
| `.get_full_structure()`       | Fetch the full structure coordinates for a given entry                                                                                                |
| `.get_symmetry_mates()`       | Compute crystal symmetry mates for a given structure                                                                                                  |
| `.get_multiple_structures()`  | Fetch data for multiple structures                                                                                                                    |


### Query Parameters

The specific set of parameters available depend on the particular `ModelQuery` method being used above. However, in general, most methods will accept the following common parameters:

| Parameter             | Description                                                           | Default                                       |
| --------------------- | --------------------------------------------------------------------- | --------------------------------------------- |
| `entry_id`            | Structure identifier (e.g., `"2HHB"`)                                 | —                                             |
| `encoding`            | Response encoding format. Supported values: `"cif"` and `"bcif"`      | `"cif"`                                       |
| `download`            | Whether to download the response to a file                            | `False`                                       |
| `filename`            | Output filename for downloaded data                                   | `None` (auto-generated from query parameters) |
| `file_directory`      | Directory where downloaded files will be saved                        | `None` (current working directory)            |
| `compress_gzip`       | Whether to gzip-compress the downloaded file                          | `False`                                       |
| `copy_all_categories` | Whether to include all metadata categories from the source entry file | `False`                                       |

Depending on the particular method being used and the level of granularity of the structure you want to fetch, one or more of the following can be specified:

| Parameter         | Description                                                 | Default | Example                         |
| ----------------- | ----------------------------------------------------------- | ------- | ------------------------------- |
| `label_entity_id` | PDB-assigned entity ID for the structural component         | `None`  | `"1"` (as in entity `4HHB_1`)   |
| `label_asym_id`   | PDB-assigned asymmetric (chain) ID                          | `None`  | `"A"` (as in instance `4HHB.A`) |
| `auth_asym_id`    | Author-assigned asymmetric (chain) ID                       | `None`  | `"A"`                           |
| `label_comp_id`   | PDB-assigned chemical component or ligand ID                | `None`  | `"HEM"`                         |
| `auth_comp_id`    | Author-assigned chemical component or ligand ID             | `None`  | `"HEM"`                         |
| `label_seq_id`    | PDB-assigned sequence number of the residue of interest     | `None`  | `123` (as in residue `A123`)    |
| `auth_seq_id`     | Author-assigned sequence number of the residue of interest  | `None`  | `123`                           |


For the complete list of possible parameters for each type of Model API query, see the [*rcsb-api* documentation](https://rcsbapi.readthedocs.io/en/latest/model_api/quickstart.html).


## Creating a Models API Query

Let us work through a couple examples/use-cases of the Models API with the `rcsbapi.model` subpackage.

### 1. Fetch the coordinates of the `HEM` ligand from PDB entry `4HHB` (first occurrence only; using `.get_ligand()`)

Note that by default this method returns only the first instance of the specified ligand (e.g., if there are 10 `HEM` ligands in the structure, only the first one is returned). If you want a specific instance of the ligand, you can specify `label_asym_id` and/or `label_entity_id`. If you want *all* occurrences of a specific ligand, you should use the `.get_atoms()` (as demonstrated in the next example).

In [ ]:
from rcsbapi.model import ModelQuery

# Fetch the first occurrence of the `HEM` ligand for entry "4HHB"
query = ModelQuery()
result = query.get_ligand(entry_id="4HHB", label_comp_id="HEM", download=True, filename="4HHB_HEM_ligand.cif", file_directory="model-output")
print(result)

### 2. Fetch the coordinates of *ALL* `HEM` instances in PDB entry `4HHB` (using `.get_atoms()`)
This can be used to fetch all `atom_site` data for a particular component using `label_comp_id` (e.g., all `HEM` ligands, all water molecules `HOH`, or all `CYS` residues), a given entity, a specific residue in the sequence, and/or a combination of these criteria.

In [ ]:
from rcsbapi.model import ModelQuery

# Fetch the metadata and `atom_site` coordinates of ALL occurrence of `HEM` in entry "4HHB"
query = ModelQuery()
result = query.get_atoms(entry_id="4HHB", label_comp_id="HEM", download=True, filename="4HHB_HEM_atoms.cif", file_directory="model-output")
print(result)

### 3. Fetch the coordinates of the surrounding residues within 5 Å of `HEM` instance `E` within PDB entry `4HHB` (using `.get_residue_interaction()`)
**Tip:** If want to fetch the coordinates of the surrounding residues for *ALL* occurrences of the ligand or residue, just provide the `label_comp_id` *without* a corresponding `label_asym_id`.

In [ ]:
from rcsbapi.model import ModelQuery

# Fetch surrounding residues for `HEM` chain `E` in entry "4HHB"
query = ModelQuery()
result = query.get_residue_interaction(
    entry_id="4HHB",
    label_comp_id="HEM",
    label_asym_id="E",
    radius=5.0,
    download=True,
    filename="4HHB_HEM_E_residue_interaction.cif",
    file_directory="model-output"
)
print(result)

### 4. Fetch the coordinates of all ligands within 5 Å of residue `A284` in PDB entry `1TQN` (using `.get_surrounding_ligands()`)

In [ ]:
from rcsbapi.model import ModelQuery

# Fetch surrounding ligands for `ALA 284` in entry "1TQN"
query = ModelQuery()
result = query.get_surrounding_ligands(
    entry_id="1TQN",
    label_comp_id="ALA",
    label_seq_id=284,
    radius=5.0,
    download=True,
    file_directory="model-output"
)
print(result)

## Further Documentation
For more extensive examples and implementation details visit our [*rcsb-api* readthedocs](https://rcsbapi.readthedocs.io/en/latest/model_api/quickstart.html) and [Model Server API specification](http://models.rcsb.org).